# Final (Ollama): the PreAct agent on the 20 dev questions, Q8_0 vs Q4_K_M, telemetry, native speed benchmark

- `preact_q8`: the PreAct agent on `qwen3.5:4b-q8_0`
- `preact_q4km`: the PreAct agent on `qwen3.5:4b`

GPU lanes (one model server each): [['preact_q8'], ['preact_q4km']]. Questions: the 20 dev questions.

Generated by a helper script; the project code is embedded below. Needs GPU T4 x2, Internet on, and the private dataset holding kit.zip.

In [ ]:
CODE = '...'
RUNS = [('preact_q8', 'qwen3.5:4b-q8_0'), ('preact_q4km', 'qwen3.5:4b')]
GPUS = 2
PARALLEL = 4
EXTRA = '--telemetry'
BASELINE = 'preact_q8'
TRACES = 2
QUESTIONS_B64 = ''
SERVER = 'ollama'
HF_MODEL = 'Qwen/Qwen3.5-4B'
SERVED_NAME = 'qwen3.5-4b'
ON_DEV = ['preact_q8', 'preact_q4km']
TOOL_PARSER = 'qwen3_coder'
VLLM_PIP = 'vllm'
SERIAL = False
CONCURRENCY = {'preact_q8': 1, 'preact_q4km': 1}
BENCH_MODELS = ['qwen3.5:4b-q8_0', 'qwen3.5:4b']

In [ ]:
import base64, glob, io, json, os, shutil, subprocess, sys, threading, time, urllib.request, zipfile
os.chdir("/kaggle/working")
zipfile.ZipFile(io.BytesIO(base64.b64decode(CODE))).extractall(".")
zips = glob.glob("/kaggle/input/**/kit.zip", recursive=True)
if zips:
    zipfile.ZipFile(zips[0]).extractall(".")
else:  # Kaggle may auto-extract an uploaded zip
    q = glob.glob("/kaggle/input/**/sql_agent_kit/data/questions_dev.json", recursive=True)[0]
    shutil.copytree(q.rsplit("/data/", 1)[0], "sql_agent_kit", dirs_exist_ok=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "langgraph", "langchain-openai", "psutil", "matplotlib"], check=True)
os.chdir("/kaggle/working/text2sql_V2")
print(subprocess.run([sys.executable, "-m", "pytest", "-q", "tests"], capture_output=True, text=True).stdout[-600:])

In [ ]:
subprocess.run("apt-get install -y -qq zstd pciutils lshw > /dev/null 2>&1", shell=True)
r = subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, capture_output=True, text=True)
print(r.stdout[-1500:], r.stderr[-1500:])
assert shutil.which("ollama"), "ollama install failed (see the installer output above)"
MODELS = sorted({model for _, model in RUNS})
URLS = []
for g in range(GPUS):  # one server per GPU: a 4B model fits on one T4, so this doubles throughput
    port = 11434 + g
    env = {**os.environ, "OLLAMA_CONTEXT_LENGTH": "16384", "OLLAMA_NUM_PARALLEL": str(PARALLEL),
           "OLLAMA_KEEP_ALIVE": "120m", "OLLAMA_HOST": f"127.0.0.1:{port}", "CUDA_VISIBLE_DEVICES": str(g)}
    subprocess.Popen(["ollama", "serve"], env=env, stdout=open(f"/kaggle/working/ollama{g}.log", "w"), stderr=subprocess.STDOUT)
    for _ in range(60):
        try:
            urllib.request.urlopen(f"http://127.0.0.1:{port}"); break
        except OSError:
            time.sleep(1)
    URLS.append(f"http://127.0.0.1:{port}/v1")
for m in MODELS:  # the servers share ~/.ollama: one pull serves both GPUs
    subprocess.run(["ollama", "pull", m], check=True, capture_output=True, env={**os.environ, "OLLAMA_HOST": "127.0.0.1:11434"})
print(URLS, subprocess.run(["ollama", "list"], capture_output=True, text=True, env={**os.environ, "OLLAMA_HOST": "127.0.0.1:11434"}).stdout)
# Warm-up: load each model on each server before any timed question, so the load time is not charged
# to the first questions (POC round 2 lost q32 that way: its first call waited about 146 s).
for url in URLS:
    for m in MODELS:
        body = json.dumps({"model": m, "messages": [{"role": "user", "content": "hi"}], "max_tokens": 1,
                           "reasoning_effort": "none"}).encode()
        req = urllib.request.Request(url + "/chat/completions", body, {"Content-Type": "application/json"})
        t = time.time()
        urllib.request.urlopen(req, timeout=600).read()
        print(f"warm-up {m} on {url}: {time.time() - t:.0f}s", flush=True)
# The assignment's "Getting started" step 3: one tool call with no framework.
print(subprocess.run([sys.executable, "scripts/hello_tool_call.py", URLS[0], MODELS[0]], capture_output=True, text=True).stdout)

In [ ]:
Q = DEV_Q = "../sql_agent_kit/data/questions_dev.json"; DBS = "../sql_agent_kit/data/databases"
if QUESTIONS_B64:
    Q = "/kaggle/working/questions.json"; open(Q, "wb").write(base64.b64decode(QUESTIONS_B64))
os.makedirs("/kaggle/working/runs", exist_ok=True)

def lane(k):  # runs k, k+n, ... go one after another to server k
    for i in range(k, len(RUNS), 1 if SERIAL else len(URLS)):
        name, model = RUNS[i]
        conc = CONCURRENCY.get(name, PARALLEL)
        q = DEV_Q if name in ON_DEV else Q  # runs marked :dev use the kit's 20 dev questions
        cmd = [sys.executable, "-m", "text2sql_v2", "--questions", q, "--db-dir", DBS, "--base-url", URLS[k],
               "--model", model, "--out", f"/kaggle/working/runs/{name}.jsonl",
               "--concurrency", str(conc)] + EXTRA.split()
        t = time.time()
        p = subprocess.run(cmd, capture_output=True, text=True)
        tail = "\n".join(p.stdout.splitlines()[-12:])  # the scorer's summary
        print(f"=== {name} on server {k} [{time.time() - t:.0f}s]\n{tail}\n{p.stderr[-1500:]}", flush=True)

threads = [threading.Thread(target=lane, args=(k,)) for k in range(1 if SERIAL else len(URLS))]
[t.start() for t in threads]; [t.join() for t in threads]

In [ ]:
for group, tag in ((set(ON_DEV), "dev"), ({n for n, _ in RUNS} - set(ON_DEV), "main")):  # one table per question set
    runs = [f"/kaggle/working/runs/{n}.jsonl" for n, _ in RUNS if n in group and os.path.exists(f"/kaggle/working/runs/{n}.jsonl")]
    if runs:
        p = subprocess.run([sys.executable, "scripts/summarize.py", *runs, "--baseline", BASELINE, "--traces", str(TRACES),
                            "--json", f"/kaggle/working/runs/summary_{tag}.json"], capture_output=True, text=True)
        print(f"##### {tag} questions", p.stdout, p.stderr[-2000:])
os.makedirs("/kaggle/working/runs/figures", exist_ok=True)
for n, _ in RUNS:  # Bonus 2: one timeline per run (agent events + telemetry on one clock)
    if os.path.isdir(f"/kaggle/working/runs/{n}_logs"):
        r = subprocess.run([sys.executable, "scripts/plot_timeline.py", f"/kaggle/working/runs/{n}_logs",
                            "--out", f"/kaggle/working/runs/figures/{n}_timeline.png"], capture_output=True, text=True)
        print(r.stdout, r.stderr[-500:])
if BENCH_MODELS and SERVER == "ollama":  # quantization: prefill/decode rates and memory from Ollama's native API
    src = next(f"/kaggle/working/runs/{n}_logs" for n, _ in RUNS if os.path.isdir(f"/kaggle/working/runs/{n}_logs"))
    r = subprocess.run([sys.executable, "scripts/bench_native.py", "--host", URLS[0][:-3], "--logs", src,
                        "--models", *BENCH_MODELS, "--n", "10", "--out", "/kaggle/working/runs/bench_native.json"],
                       capture_output=True, text=True)
    print(r.stdout, r.stderr[-1500:])
shutil.make_archive("/kaggle/working/runs_bundle", "zip", "/kaggle/working/runs")
print(os.path.getsize("/kaggle/working/runs_bundle.zip"), "bytes in runs_bundle.zip")